# Chapter 15 Lab — A Minimal RAG Pipeline

Chunk -> embed -> index -> retrieve -> generate, end to end, on a small public-domain document
collection. This reimplements, cleanly with public libraries and redistributable data, the same
pipeline shape used in the author's own AI-tutor course project — not any client-specific files
from that project.

## 1. A small source document collection (self-written for this lab)

In [ ]:
documents = [
    "The Transformer architecture, introduced in 2017, replaced recurrent networks with "
    "self-attention, letting every token attend to every other token in parallel.",
    "BERT (2018) pretrains a Transformer encoder with a masked-language-modeling objective, "
    "producing strong contextual representations for understanding tasks.",
    "GPT models are decoder-only Transformers trained with a causal next-token objective, "
    "which makes them natural text generators and, at scale, in-context learners.",
    "Retrieval-Augmented Generation (RAG), introduced in 2020, retrieves relevant passages at "
    "query time and conditions a generative model's output on them, reducing hallucination and "
    "removing the need to retrain the model when source documents change.",
    "A RAG system's quality depends on both its retriever (does it find the right passage?) "
    "and its generator (does it use that passage correctly?) — the two must be evaluated "
    "separately to know which one to fix.",
]

## 2. Chunking (with overlap)

In [ ]:
def chunk_text(text, chunk_size=120, overlap=20):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + chunk_size]))
        i += chunk_size - overlap
    return chunks

chunks = [c for doc in documents for c in chunk_text(doc, chunk_size=25, overlap=5)]
print(f"{len(documents)} documents -> {len(chunks)} chunks")

## 3. Embedding + indexing with FAISS

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embs = embedder.encode(chunks, convert_to_numpy=True)
index = faiss.IndexFlatIP(chunk_embs.shape[1])
faiss.normalize_L2(chunk_embs)
index.add(chunk_embs)
print("FAISS index size:", index.ntotal)

## 4. Retrieval

In [ ]:
def retrieve(query, k=2):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, k)
    return [chunks[i] for i in idxs[0]]

print(retrieve("What problem does RAG solve?"))

## 5. Augmented generation

In [ ]:
from transformers import pipeline
generator = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)
    return [{"role": "system", "content":
             "Answer the question using ONLY the provided context. "
             "If the answer is not in the context, say 'I don't know.'"},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]

def rag_answer(question, k=2):
    ctx = retrieve(question, k=k)
    out = generator(build_prompt(question, ctx), max_new_tokens=60)
    return out[0]["generated_text"][-1]["content"], ctx

answer, ctx = rag_answer("What problem does RAG solve?")
print("Retrieved context:", ctx)
print("Answer:", answer)

## 6. Grounded refusal — an out-of-context question

In [ ]:
answer, ctx = rag_answer("What is the capital of Kazakhstan?")
print("Retrieved context:", ctx)
print("Answer:", answer)
print("\nCompare this to asking the bare generator the same question with no retrieval — it will\n"
      "likely answer anyway from parametric memory, illustrating exactly the grounding gap RAG closes.")

## Exercise

Add three more documents on a topic *not* covered above, re-index, and ask a question about one
of them. Then ask a question that is answerable only by *combining* facts from two different
chunks — does the top-`k` retrieval reliably return both? What does that suggest about choosing
`k`?